In [18]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
sys.path.append("..")
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [19]:
%time train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

CPU times: user 27.4 ms, sys: 7.04 ms, total: 34.4 ms
Wall time: 42.5 ms
CPU times: user 15.3 ms, sys: 12.1 ms, total: 27.3 ms
Wall time: 45.1 ms
CPU times: user 7.08 ms, sys: 2.01 ms, total: 9.09 ms
Wall time: 15.7 ms


In [20]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [21]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [22]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [23]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [24]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [25]:
epochs = 150
early_stop_metric = "ema_qini"
ema = True
ema_alpha = 0.25
patience = 20
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
print (f" epochs = {epochs}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 early stop = ema_qini
 use ema = True
 ema alpha = 0.25
 patience = 20
 early stop start = 30


In [26]:
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 1e-5, 3e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    grid_outcome_dropout = trial.suggest_float("outcome_dropout", 0.0, 0.5)
    grid_shared_dropout = trial.suggest_float("shared_dropout", 0.0, 0.5)
    grid_outcome_hidden = trial.suggest_int("outcome_hidden", 50, 400, step=50)
    grid_shared_hidden = trial.suggest_int("shared_hidden", 50, 400, step=50)
    grid_beta = trial.suggest_float("beta", 0.0, 10)
    grid_alpha = trial.suggest_float("alpha", 0.0, 10)

    val_auqc_list = []
    val_ate_err_list = []
    for SEED in seeds:
        seed_everything(SEED)

        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=grid_shared_hidden,
            outcome_hidden=grid_outcome_hidden,
            outcome_dropout=grid_outcome_dropout,
            shared_dropout=grid_shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            alpha = grid_alpha,
            beta= grid_beta
        )

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            dragonnet.fit(train_loader, val_loader)

        x_men_test_t_on_device = x_men_test_t.to(device)
        y0_pred, y1_pred = dragonnet.predict(x_men_test_t_on_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()
        
        current_val_auqc = auqc(y_true, t_true, uplift_pred, bins=100, plot=False)
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
        current_val_ate_err = abs(ate_pred - ate_true)

        val_auqc_list.append(current_val_auqc)
        val_ate_err_list.append(current_val_ate_err)

    # Calculate aggregated metrics across the 5 validation seeds
    mean_auqc = float(np.mean(val_auqc_list))
    std_auqc = float(np.std(val_auqc_list))
    mean_ate_err = float(np.mean(val_ate_err_list))

    # Apply penalty for instability and miscalibration
    penalty_std = std_auqc * 1.0
    penalty_ate = mean_ate_err * 0.05

    final_score = mean_auqc - penalty_std - penalty_ate

    trial.set_user_attr("mean_val_auqc", mean_auqc)
    trial.set_user_attr("std_Val_auqc", std_auqc)
    trial.set_user_attr("mean_val_ate_err", mean_ate_err)
    trial.set_user_attr("final_score", final_score)
    return final_score

def print_trial_callback(study, trial):
    value = float(trial.value) if trial.value is not None else float("nan")
    best_trial = study.best_trial
    best_value = float(best_trial.value) if best_trial.value is not None else float("nan")
    print(
        f"Finished trial {trial.number}: val loss: {value:.4f} - "
        f"with hyperparameters: {trial.params} | "
        f"best trial: {best_trial.number} loss: {best_value:.4f}",
        flush=True
    )

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[print_trial_callback])

trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": round(float(t.params["lr"]), 4),
        "weight_decay": round(float(t.params["weight_decay"]), 4),
        "shared_hidden": int(t.params["shared_hidden"]),
        "outcome_hidden": int(t.params["outcome_hidden"]),
        "shared_dropout": round(float(t.params["shared_dropout"]), 3),
        "outcome_dropout": round(float(t.params["outcome_dropout"]), 3),
        "alpha": round(float(t.params["alpha"]), 3),
        "beta": round(float(t.params["beta"]), 3),
        "mean_val_auqc": float(t.value),
        "std_Val_auqc": float(t.user_attrs.get("std_Val_auqc", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_val_auqc", ascending=True).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "shared_hidden": int(best_params["shared_hidden"]),
    "outcome_hidden": int(best_params["outcome_hidden"]),
    "shared_dropout": float(best_params["shared_dropout"]),
    "outcome_dropout": float(best_params["outcome_dropout"]),
    "alpha": float(best_params["alpha"]),
    "beta": float(best_params["beta"]),
    "mean_Val_auqc": float(study.best_value),
    "std_Val_auqc": float(study.best_trial.user_attrs.get("std_Val_auqc", np.nan))
})

  0%|          | 0/50 [00:26<?, ?it/s]


[W 2026-04-01 14:57:59,752] Trial 0 failed with parameters: {'lr': 5.070624901428979e-05, 'weight_decay': 0.00013625464993948335, 'outcome_dropout': 0.2041371296334657, 'shared_dropout': 0.2216925330468017, 'outcome_hidden': 250, 'shared_hidden': 350, 'beta': 9.347555747146833, 'alpha': 9.788862701608304} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_28308/4126071723.py", line 48, in objective
    dragonnet.fit(train_loader, val_loader)
  File "/home/ducvu0904/Documents/Lab/RERUM/dragonnet/dragonnet.py", line 80, in fit
    for x_batch , t_batch ,y_batch in train_loader:
  File "/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
         

KeyboardInterrupt: 

In [ ]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 40
Best mean_Val_AUQC: 0.536263
Best params: {'lr': 3.250632060616809e-05, 'weight_decay': 4.8270498289769964e-05, 'outcome_dropout': 0.29654695898615496, 'shared_dropout': 0.12267178350366371, 'outcome_hidden': 350, 'shared_hidden': 350, 'beta': 4.81328807391155, 'alpha': 9.69415290337158}

Best config table:


,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,alpha,beta,mean_Val_auqc,std_Val_auqc
0,0.000033,0.000048,350.0,350.0,0.122672,0.296547,9.694153,4.813288,0.536263,0.012414



Top 10 trials:


,trial,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,alpha,beta,mean_val_auqc,std_Val_auqc
0,39,0.0001,0.0000,400,250,0.031,0.262,7.245,8.514,0.249918,0.103778
1,49,0.0000,0.0001,300,300,0.028,0.003,7.811,5.668,0.271336,0.117238
2,9,0.0000,0.0001,300,350,0.161,0.181,5.129,9.998,0.279878,0.117518
3,21,0.0000,0.0004,250,250,0.109,0.390,4.740,1.134,0.281684,0.146448
4,13,0.0000,0.0000,250,200,0.360,0.340,1.864,3.201,0.298409,0.137349
5,14,0.0001,0.0000,400,300,0.137,0.486,8.951,7.591,0.313523,0.098210
6,43,0.0000,0.0001,300,400,0.064,0.299,8.403,4.822,0.321929,0.131951
7,38,0.0000,0.0001,300,400,0.058,0.282,8.462,7.018,0.342534,0.132180
8,24,0.0000,0.0037,150,350,0.019,0.438,6.116,4.392,0.348976,0.134884
9,15,0.0000,0.0014,150,300,0.314,0.425,5.741,5.385,0.349122,0.104924


In [28]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on validation and test sets (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []
val_auqc_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_shared_hidden = int(best_cfg['shared_hidden'])
best_outcome_hidden = int(best_cfg['outcome_hidden'])
best_shared_dropout = float(best_cfg['shared_dropout'])
best_outcome_dropout = float(best_cfg['outcome_dropout'])
best_alpha = float(best_cfg['alpha'])
best_beta = float(best_cfg['beta'])

print("Evaluating with best validation config:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"  shared_hidden={best_shared_hidden}, outcome_hidden={best_outcome_hidden}")
print(f"  shared_dropout={best_shared_dropout:.3f}, outcome_dropout={best_outcome_dropout:.3f}")
print(f"  alpha={best_alpha:.3f}, beta={best_beta:.3f}")
print(f"Number of seeds: {len(seeds)}")

# 2. Loop over seeds for robust validation/test evaluation
for SEED in seeds:
    seed_everything(SEED)

    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=best_shared_hidden,
        outcome_hidden=best_outcome_hidden,
        outcome_dropout=best_outcome_dropout,
        shared_dropout=best_shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        alpha=best_alpha,
        beta=best_beta
    )

    dragonnet.fit(train_loader, val_loader)

    # Validation prediction for mean validation AUQC across seeds
    x_men_val_t_on_device = x_men_val_t.to(device)
    y0_val_pred, y1_val_pred = dragonnet.predict(x_men_val_t_on_device)
    uplift_val_pred = (y1_val_pred - y0_val_pred).detach().cpu().numpy().flatten()
    y_val_true = y_men_val_t.detach().cpu().numpy().flatten()
    t_val_true = t_men_val_t.detach().cpu().numpy().flatten()
    current_val_auqc = auqc(y_val_true, t_val_true, uplift_val_pred, bins=100, plot=False)
    val_auqc_runs.append(current_val_auqc)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = dragonnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED} | Val AUQC={current_val_auqc:.4f}")

# 3. Aggregate validation AUQC and final test metrics
avg_val_auqc = float(np.mean(val_auqc_runs))
std_val_auqc = float(np.std(val_auqc_runs))
print("\n" + "=" * 85)
print(f"Mean validation AUQC across 5 seeds: {avg_val_auqc:.4f} ± {std_val_auqc:.4f}")
print("=" * 85)

df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Evaluating with best validation config:
  lr=3.3e-05, weight_decay=4.8e-05
  shared_hidden=350, outcome_hidden=350
  shared_dropout=0.123, outcome_dropout=0.297
  alpha=9.694, beta=4.813
Number of seeds: 5
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.25)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Base Loss: 257.3862 | Tarreg Loss: 608.389404 | Total Loss: 865.7756 | Val Loss: 505.4324 | Val Qini: 0.5406 | EMA Qini: 0.5406 | Best EMA: 0.5406 ⭐ NEW BEST EMA
Epoch 2/150 | Base Loss: 424.5822 | Tarreg Loss: 1009.060852 | Total Loss: 1433.6431 | Val Loss: 505.3989 | Val Qini: 0.6399 | EMA Qini: 0.5654 | Best EMA: 0.5654 ⭐ NEW BEST EMA
Epoch 3/150 | Base Loss: 402.5339 | Tarreg Loss: 954.271912 | Total Loss: 1356.8058 | Val Loss: 505.3650 | Val Qini: 0.6891 | EMA Qini: 0.5964 | Best EMA: 0.5964 ⭐ NEW BEST EMA
Epoch 4/150 | Base Loss: 343.2242 | Tarreg Loss: 816